# Step 02: Data Validation Rules & Schema Checks

## Overview
This notebook defines and executes explicit data validity assertions across all five raw datasets before data cleaning.

### Validation Categories:
1. **Schema Check**: Required columns exist in raw files.
2. **Type Check**: Columns match expected numeric/datetime/string data types.
3. **Range Check**: Numerical values fall within valid physical/business boundaries (e.g. Age 18–100, scores 1–5).
4. **Uniqueness Check**: Primary key identifiers contain no duplicate records.
5. **Category Check**: Categorical fields contain allowed enumerated values (e.g., `Attrition` in `['Yes', 'No']`).

> Note: Rules flagged with `[API Schema]` will be refactored into Pydantic validators in `app/validation/employee_schema.py` during Day 4.


In [1]:
import pandas as pd
import numpy as np
import os

RAW_DIR = os.path.join("..", "data", "raw")


---
## 1. Employee Attrition Data Validation (`employee_attrition.csv`)


In [2]:
attr_df = pd.read_csv(os.path.join(RAW_DIR, "employee_attrition.csv"))

# 1.1 Schema Check
expected_attr_cols = {'Age', 'Attrition', 'Department', 'DistanceFromHome', 'Education', 
                      'EmployeeNumber', 'Gender', 'JobRole', 'MonthlyIncome', 'OverTime', 
                      'WorkLifeBalance', 'YearsAtCompany', 'YearsSinceLastPromotion'}
assert expected_attr_cols.issubset(set(attr_df.columns)), f"Missing columns in Attrition dataset: {expected_attr_cols - set(attr_df.columns)}"
print("✔ Schema Check Passed: All required columns exist.")

# 1.2 Type Check [API Schema]
assert pd.api.types.is_integer_dtype(attr_df['Age']), "Age must be integer type"
assert pd.api.types.is_integer_dtype(attr_df['MonthlyIncome']), "MonthlyIncome must be integer type"
assert pd.api.types.is_object_dtype(attr_df['Attrition']), "Attrition must be string/object type"
print("✔ Type Check Passed: Column data types are correct.")

# 1.3 Range Check [API Schema]
assert attr_df['Age'].between(18, 100).all(), "Age must be between 18 and 100"
assert (attr_df['MonthlyIncome'] > 0).all(), "MonthlyIncome must be positive"
assert attr_df['DistanceFromHome'].between(1, 100).all(), "DistanceFromHome out of bounds"
print("✔ Range Check Passed: Age (18-100), MonthlyIncome (>0), DistanceFromHome (1-100).")

# 1.4 Uniqueness Check
assert attr_df['EmployeeNumber'].is_unique, "EmployeeNumber must be unique"
print("✔ Uniqueness Check Passed: EmployeeNumber is unique.")

# 1.5 Category Check [API Schema]
allowed_attrition = {'Yes', 'No'}
actual_attrition = set(attr_df['Attrition'].unique())
assert actual_attrition.issubset(allowed_attrition), f"Invalid Attrition categories found: {actual_attrition - allowed_attrition}"

allowed_overtime = {'Yes', 'No'}
assert set(attr_df['OverTime'].unique()).issubset(allowed_overtime), "Invalid OverTime values"
print("✔ Category Check Passed: Attrition in {'Yes', 'No'}, OverTime in {'Yes', 'No'}.")


✔ Schema Check Passed: All required columns exist.
✔ Type Check Passed: Column data types are correct.
✔ Range Check Passed: Age (18-100), MonthlyIncome (>0), DistanceFromHome (1-100).
✔ Uniqueness Check Passed: EmployeeNumber is unique.
✔ Category Check Passed: Attrition in {'Yes', 'No'}, OverTime in {'Yes', 'No'}.


---
## 2. Performance & Engagement Data Validation (`hr_performance_engagement.csv`)


In [3]:
perf_df = pd.read_csv(os.path.join(RAW_DIR, "hr_performance_engagement.csv"))

# 2.1 Schema Check
expected_perf_cols = {'Employee ID', 'Performance Score', 'Engagement Score', 'Satisfaction Score', 'Work-Life Balance Score', 'Current Employee Rating'}
assert expected_perf_cols.issubset(set(perf_df.columns)), "Missing columns in Performance dataset"
print("✔ Schema Check Passed: Performance columns exist.")

# 2.2 Range Check [API Schema]
assert perf_df['Engagement Score'].between(1, 5).all(), "Engagement Score must be between 1 and 5"
assert perf_df['Satisfaction Score'].between(1, 5).all(), "Satisfaction Score must be between 1 and 5"
assert perf_df['Work-Life Balance Score'].between(1, 5).all(), "Work-Life Balance Score must be between 1 and 5"
assert perf_df['Current Employee Rating'].between(1, 5).all(), "Current Employee Rating must be between 1 and 5"
print("✔ Range Check Passed: Engagement, Satisfaction, Work-Life Balance scores all within 1–5 scale.")

# 2.3 Uniqueness Check
assert perf_df['Employee ID'].is_unique, "Employee ID must be unique in performance dataset"
print("✔ Uniqueness Check Passed: Employee ID is unique.")


✔ Schema Check Passed: Performance columns exist.
✔ Range Check Passed: Engagement, Satisfaction, Work-Life Balance scores all within 1–5 scale.
✔ Uniqueness Check Passed: Employee ID is unique.


---
## 3. Occupation Data Validation (`occupation_data.csv`)


In [4]:
occ_df = pd.read_csv(os.path.join(RAW_DIR, "occupation_data.csv"))

assert {'O*NET-SOC Code', 'Title', 'Description'}.issubset(set(occ_df.columns)), "Missing Occupation columns"
assert occ_df['O*NET-SOC Code'].is_unique, "O*NET-SOC Code must be unique in occupation master"
assert not occ_df['Title'].isnull().any(), "Occupation titles must not be null"
print("✔ Occupation Master Validation Passed: Unique SOC codes and non-null titles.")


✔ Occupation Master Validation Passed: Unique SOC codes and non-null titles.


---
## 4. Essential Skills Data Validation (`essential_skills.csv`)


In [5]:
ess_df = pd.read_csv(os.path.join(RAW_DIR, "essential_skills.csv"))

assert {'O*NET-SOC Code', 'Element Name', 'Data Value', 'Scale ID'}.issubset(set(ess_df.columns)), "Missing Essential Skills columns"
assert (ess_df['Data Value'] >= 0).all(), "Skill Data Value must be non-negative"
assert set(ess_df['Scale ID'].unique()).issubset({'IM', 'LV'}), "Scale ID must be IM (Importance) or LV (Level)"
print("✔ Essential Skills Validation Passed: Scale IDs in {'IM', 'LV'}, Data Value >= 0.")


✔ Essential Skills Validation Passed: Scale IDs in {'IM', 'LV'}, Data Value >= 0.


---
## 5. Software Skills Data Validation (`software_skills.csv`)


In [6]:
soft_df = pd.read_csv(os.path.join(RAW_DIR, "software_skills.csv"))

assert {'O*NET-SOC Code', 'Workplace Example', 'Hot Technology', 'In Demand'}.issubset(set(soft_df.columns)), "Missing Software Skills columns"
assert set(soft_df['Hot Technology'].unique()).issubset({'Y', 'N'}), "Hot Technology must be Y or N"
assert set(soft_df['In Demand'].unique()).issubset({'Y', 'N'}), "In Demand must be Y or N"
print("✔ Software Skills Validation Passed: Hot Technology & In Demand flags in {'Y', 'N'}.")


✔ Software Skills Validation Passed: Hot Technology & In Demand flags in {'Y', 'N'}.


---
## Validation Rules to Migrate to `app/validation/employee_schema.py`

The following checks will be enforced dynamically via Pydantic models in FastAPI:
1. `Age`: Integer, 18 <= Age <= 100
2. `MonthlyIncome`: Integer/Float > 0
3. `Engagement Score`, `Satisfaction Score`, `WorkLifeBalance`: Integer between 1 and 5
4. `Attrition`: Enum string (`Yes` / `No`)
5. `OverTime`: Enum string (`Yes` / `No`)
6. `DistanceFromHome`: Integer >= 0
